# Probability-of-default model (LightGBM classifier)

**Algorithm switch from the PyTorch MLP version.** Gradient-boosted trees (LightGBM/XGBoost/CatBoost) are the standard choice for tabular credit-risk models in industry — they typically beat plain feedforward nets on this kind of data, need far less tuning (no architecture search, no learning-rate schedule), handle categoricals natively, don't need feature scaling, and — the concrete win for this codebase — support `shap.TreeExplainer`: **exact**, fast Shapley values, the same explanation method already used for the credit-score model. The previous PyTorch version needed an approximate `KernelExplainer` with a synthetic background sample (see `build_default_pd_background` in `mlPredictor.py`) purely because it was a neural net; that whole code path becomes unnecessary once both models are tree-based.

Feature set is unchanged from the previous version — `Age, Income, LoanAmount, CreditScore, InterestRate, LoanTerm, DTIRatio, HasMortgage, HasDependents, LoanPurpose` — matching what the live loan application form actually collects (`Backend/src/schemas/loanSchemas.js`). The original notebook this was based on dropped `MonthsEmployed`, `NumCreditLines`, `Education`, `EmploymentType`, `MaritalStatus`, `HasCoSigner` without explanation; those are plausible real default-risk predictors, but the live form doesn't collect them, so adding them here would require a separate, larger change to the intake form/schema/route — out of scope for this algorithm swap.

No `StandardScaler` this time — tree-based models split on raw feature values and are invariant to monotonic transforms, so scaling is genuinely unnecessary (not just harmless) here, unlike the PyTorch version which needed it.

**Not yet run** — `Loan_default.csv` must be present in this directory.
**Update: `CreditScore` removed as an input feature.** The live app chains this model's `CreditScore` input from the credit-score model's own prediction (`handle_predict` in `mlPredictor.py` calls `predict_credit_score()` first, then passes its output into this model). That's a real train/serve skew: this notebook was training against the dataset's ground-truth `CreditScore` column, not the near-random values the live credit-score model actually produces (see its own notebook -- R² ≈ 0, `CreditScore` is uncorrelated with every other field in this dataset). Feeding a near-signal-free prediction into another model as a feature doesn't corrupt it outright, but every metric measured with the real ground-truth column was optimistic relative to what the live pipeline can actually deliver. Dropping it here is the honest fix until the credit-score model has real data to train on.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    accuracy_score, brier_score_loss,
)


In [2]:
df = pd.read_csv('Loan_default.csv')
df.drop(columns=['LoanID', 'MonthsEmployed', 'NumCreditLines', 'Education',
                  'EmploymentType', 'MaritalStatus', 'HasCoSigner'], inplace=True)


In [3]:
df['HasMortgage'] = df['HasMortgage'].replace(['Yes', 'No'], [1, 0]).astype(int)
df['HasDependents'] = df['HasDependents'].replace(['Yes', 'No'], [1, 0]).astype(int)
df['LoanPurpose'] = df['LoanPurpose'].replace(
    ['Business', 'Home', 'Education', 'Other', 'Auto'], [0, 1, 2, 3, 4]
).astype('category')  # native categorical support, unlike the ordinal-int NN input


## Select features/target by name, check class balance, split
Same stratified train/val/test split as before, but no scaler step — LightGBM doesn't need one.

In [4]:
feature_cols = ['Age', 'Income', 'LoanAmount', 'InterestRate',
                'LoanTerm', 'DTIRatio', 'HasMortgage', 'HasDependents', 'LoanPurpose']
target_col = 'Default'

X = df[feature_cols]
y = df[target_col]

print('Class balance:')
print(y.value_counts(normalize=True))

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)


Class balance:
Default
0    0.883872
1    0.116128
Name: proportion, dtype: float64


## Train with class-imbalance weighting and early stopping
`scale_pos_weight` is LightGBM's native equivalent of the PyTorch version's `pos_weight` — same purpose (upweight the minority/default class in the loss), same calibration side effect. It's corrected the same way: isotonic calibration fit on validation predictions, then the decision threshold is tuned on the *calibrated* validation probabilities, never on test.

In [5]:
train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=['LoanPurpose'])
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data,
                        categorical_feature=['LoanPurpose'])

n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()
scale_pos_weight = n_neg / max(n_pos, 1)
print(f'scale_pos_weight = {scale_pos_weight:.2f}  (train split: {n_neg} neg / {n_pos} pos)')

params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    # NOT binary_logloss: scale_pos_weight distorts the loss surface, so logloss can
    # look like it's getting worse within a few rounds even while ranking (AUC) keeps
    # improving -- early stopping on logloss triggered after just 2 rounds here
    # (underfit). AUC is rank-based and unaffected by the class-weight distortion.
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "scale_pos_weight": scale_pos_weight,
    "verbose": -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(50)],
)


scale_pos_weight = 7.61  (train split: 144443 neg / 18978 pos)
Training until validation scores don't improve for 100 rounds


[50]	valid_0's auc: 0.725104


[100]	valid_0's auc: 0.724654


[150]	valid_0's auc: 0.723552
Early stopping, best iteration is:
[62]	valid_0's auc: 0.725456


## Calibrate on validation, tune the decision threshold

In [6]:
val_probs_raw = model.predict(X_val, num_iteration=model.best_iteration)

calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(val_probs_raw, y_val)
val_probs_calibrated = calibrator.predict(val_probs_raw)

thresholds = np.linspace(0.01, 0.99, 99)
f1_scores = [f1_score(y_val, (val_probs_calibrated > t).astype(int)) for t in thresholds]
best_threshold = thresholds[int(np.argmax(f1_scores))]
print(f'Tuned decision threshold (max F1 on calibrated validation probs): {best_threshold:.2f}')


Tuned decision threshold (max F1 on calibrated validation probs): 0.17


## Final evaluation — test set touched exactly once, here

In [7]:
test_probs_raw = model.predict(X_test, num_iteration=model.best_iteration)
test_probs = calibrator.predict(test_probs_raw)

print(f"Test AUC-ROC:   {roc_auc_score(y_test, test_probs):.4f}")
print(f"Test PR-AUC:    {average_precision_score(y_test, test_probs):.4f}")
print(f"Test Brier:     {brier_score_loss(y_test, test_probs):.4f}  (lower is better-calibrated)")
print()
print(f"@0.5 threshold           accuracy={accuracy_score(y_test, (test_probs > 0.5).astype(int)):.4f}  "
      f"f1={f1_score(y_test, (test_probs > 0.5).astype(int)):.4f}")
print(f"@{best_threshold:.2f} threshold (tuned)  accuracy={accuracy_score(y_test, (test_probs > best_threshold).astype(int)):.4f}  "
      f"f1={f1_score(y_test, (test_probs > best_threshold).astype(int)):.4f}")


Test AUC-ROC:   0.7301
Test PR-AUC:    0.2847
Test Brier:     0.0934  (lower is better-calibrated)

@0.5 threshold           accuracy=0.8857  f1=0.0996
@0.17 threshold (tuned)  accuracy=0.7685  f1=0.3403


## Exact SHAP explanation via TreeExplainer
No background sample needed — same exact-Shapley-value method already used for the credit-score model, replacing the previous version's approximate `KernelExplainer` + synthetic marginal-normal background sample.

In [8]:
import shap

explainer = shap.TreeExplainer(model)
sample = X_test.iloc[:1]
shap_values = explainer(sample)

for name, val in zip(feature_cols, shap_values.values[0]):
    print(f'{name:20s} {val:+.4f}')


/Users/viveksawant/Desktop/CreditSure/.venv-creditsure/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Age                  -0.4338
Income               +0.6950
LoanAmount           -0.1951
InterestRate         -0.2729
LoanTerm             +0.0001
DTIRatio             +0.0236
HasMortgage          -0.0477
HasDependents        -0.0529
LoanPurpose          +0.0300


## Save model + calibrator + preprocessing artifact
Saved under versioned `_v2` filenames, distinct from the live `models/default_model.pkl` path `mlPredictor.py` currently loads — wiring this in is a separate, deliberate serving-code change, not a side effect of running this notebook.

In [9]:
import json
import joblib
from pathlib import Path

out_dir = Path('../models')
out_dir.mkdir(exist_ok=True)
model.save_model(str(out_dir / 'default_model_v2.txt'))
joblib.dump(calibrator, out_dir / 'default_pd_calibrator_v2.joblib')

artifact_path = Path('../ml_artifacts/preprocessing_v2.json')
existing = json.loads(artifact_path.read_text()) if artifact_path.exists() else {}

existing['version'] = 'v2'
existing.setdefault('default_pd', {})
existing['default_pd']['algorithm'] = 'lightgbm'
existing['default_pd']['numerical_impute'] = {
    'age': 40, 'income': 60000, 'loanAmount': 20000, 'loanRate': 10,
    'loanTerm': 36, 'existingDebtPayment': 500, 'creditScore': 650,
}
existing['default_pd']['categorical_mapping'] = {
    'loanPurpose': {
        'Business': 0, 'Home': 1, 'Education': 2,
        'Other': 3, 'Others': 3, 'Auto': 4, 'Automobile': 4,
    }
}
existing['default_pd']['feature_order'] = feature_cols
existing['default_pd'].pop('scaler', None)  # no scaler needed for a tree model
existing['default_pd']['decision_threshold'] = float(best_threshold)

artifact_path.write_text(json.dumps(existing, indent=2))
print(f"Wrote {artifact_path}")


Wrote ../ml_artifacts/preprocessing_v2.json
